In [2]:
import os
import numpy as np
import pickle
import h5py
#import pandas as pd

from __future__ import print_function
import keras
from keras.datasets.cifar10 import load_data
from keras import layers, models
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D



2026-04-08 11:34:01.337684: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775615641.348880   21796 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775615641.352444   21796 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 11:34:01.365356: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
classes = 10
current_path = os.path.join(os.getcwd(), 'current_model')
#print(current_path)
#model_name = 'cifar100.h5'
(x_train, y_train) , (x_test, y_test) = load_data()
print('x_train_dims : ' , x_train.shape)
print('x_test_dims : ', x_test.shape)
print('y_train_dims : ', y_train.shape)
print('y_test_dims : ', y_test.shape)

print( 'number of training examples available : ', x_train.shape[0])
print('number of testing examples available : ', x_test.shape[0])

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

x_train /= 255.0
x_test /= 255.0

x_train_dims :  (50000, 32, 32, 3)
x_test_dims :  (10000, 32, 32, 3)
y_train_dims :  (50000, 1)
y_test_dims :  (10000, 1)
number of training examples available :  50000
number of testing examples available :  10000


# 하이퍼 파라미터 변경
- minibatch수
- fully connection 개수
- epoch
- data 많을수록 좋다
- filter수, filter크기
- conv수 (conv +relu + pool + conv + pool + ... ), Global pooling(flatten전에 한번)
- Batch Normalization layer 


In [ ]:
# 모델
model = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1),

    # Block 1
    layers.Conv2D(32, (3,3), padding='same', use_bias=False, input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(32, (3,3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(64, (3,3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),

    # Block 3
    layers.Conv2D(128, (3,3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(128, (3,3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.4),

    layers.GlobalAveragePooling2D(),

    # Classifier
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(10)  # CIFAR-10
])

# 컴파일
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# 콜백
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-5
)

early_stop = keras.callbacks.EarlyStopping(
    patience=15,
    restore_best_weights=True
)

# 학습
history = model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[lr_scheduler, early_stop],
    verbose=1
)

# 컴파일
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
# 학습 (더 많은 epoch 권장)
history_final = model.fit(
    x_train, y_train, 
    epochs=20,  # epoch 증가
    validation_data=(x_test, y_test),
    batch_size=64  # 배치 크기 조정
)

Epoch 1/100


E0000 00:00:1775616109.626309   21796 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1_1/dropout_4_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - accuracy: 0.3551 - loss: 1.8169 - val_accuracy: 0.4670 - val_loss: 1.4626 - learning_rate: 0.0010
Epoch 2/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.4937 - loss: 1.4014 - val_accuracy: 0.5192 - val_loss: 1.3509 - learning_rate: 0.0010
Epoch 3/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.5427 - loss: 1.2808 - val_accuracy: 0.5922 - val_loss: 1.1849 - learning_rate: 0.0010
Epoch 4/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.5794 - loss: 1.1915 - val_accuracy: 0.5020 - val_loss: 1.5009 - learning_rate: 0.0010
Epoch 5/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.6029 - loss: 1.1312 - val_accuracy: 0.5612 - val_loss: 1.2896 - learning_rate: 0.0010
Epoch 6/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.6268 - loss: 1.0702 - val_accuracy: 0.5720 - val_loss: 1.3211 - learning_rate: 0.0010
Epoch 7/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.6423 - loss: 

KeyboardInterrupt: 

In [2]:
import pandas as pd

# CSV 로드
df = pd.read_csv("/home/pcsl/Documents/plecs/sepic/plecs_python_auto/experiment_results.csv")

# best_val_accuracy 기준 내림차순 정렬 후 상위 5개
top5 = df.sort_values(by="best_val_accuracy", ascending=False).head(5)

# 보고 싶은 컬럼만 선택
cols = [
    "batch_size",
    "epochs",
    "learning_rate",
    "num_conv_blocks",
    "filters",
    "kernel_size",
    "use_batchnorm",
    "dropout_rate",
    "dense_units"
]

print(top5[cols])

     batch_size  epochs  learning_rate  num_conv_blocks    filters  \
43           32      50         0.0010                3  32_64_128   
139          32     100         0.0010                3  32_64_128   
90           32      50         0.0005                3  32_64_128   
138          32     100         0.0010                3  32_64_128   
187          32     100         0.0005                3  32_64_128   

     kernel_size  use_batchnorm  dropout_rate dense_units  
43             5           True           0.3       64_32  
139            5           True           0.3       64_32  
90             5           True           0.3          32  
138            5           True           0.3          32  
187            5           True           0.3       64_32  


In [10]:
epochs = 4
data_augmentation = False
predictions = 20
batch_size = 64
validation = []

for i in  range(epochs):
    if not data_augmentation:
        print("************We are not using Data Augmentation************")
        model.fit(x_train, y_train, batch_size= batch_size, epochs = epochs, validation_data = (x_test, y_test), shuffle = True)
    else:
        print("*************Show something else here **********")
        
pickle.dump(validation, open("loss_validation.p",'wb'))

************We are not using Data Augmentation************
Epoch 1/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6376 - loss: 1.0403 - val_accuracy: 0.6800 - val_loss: 0.9131
Epoch 2/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6547 - loss: 0.9936 - val_accuracy: 0.7122 - val_loss: 0.8205
Epoch 3/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6683 - loss: 0.9535 - val_accuracy: 0.7193 - val_loss: 0.8091
Epoch 4/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6787 - loss: 0.9286 - val_accuracy: 0.7294 - val_loss: 0.7818
************We are not using Data Augmentation************
Epoch 1/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6916 - loss: 0.8961 - val_accuracy: 0.7369 - val_loss: 0.7622
Epoch 2/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6952 - loss: 0.8818 - val_accuracy: 0.7393 - val_loss: 0.7501
Epoch 3/4
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7018 - loss: 0.8581 - val_accuracy: 0.7463 - val_loss: 